# LangChain Memory 시리즈 7/7: VectorStoreRetrieverMemory

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- 임베딩(Embedding)과 벡터 유사도 검색의 원리를 설명할 수 있어요
- FAISS 벡터 스토어를 초기화하고 `VectorStoreRetrieverMemory`를 구성할 수 있어요
- `k` 값으로 검색 결과 개수를 조절할 수 있어요
- 순서 기반 메모리와 의미 기반 메모리의 차이를 비교하고 선택 기준을 세울 수 있어요
- Memory 시리즈 7종 전체를 비교해 서비스 요건에 맞는 메모리를 선택할 수 있어요

## 사전 지식

| 개념 | 설명 |
|------|------|
| 임베딩 (Embedding) | 텍스트를 의미를 담은 숫자 벡터로 변환하는 기법 |
| 벡터 유사도 (Vector Similarity) | 두 벡터 간 코사인 유사도 등을 계산해 의미적 가까움을 측정 |
| FAISS | Meta가 만든 빠른 벡터 검색 라이브러리 |
| RAG (Retrieval-Augmented Generation) | 검색으로 찾은 문서를 LLM 입력에 추가하는 패턴 (6_RAG 섹션 참고) |

## 순서 기반 vs 의미 기반 메모리

1~6번 노트북의 메모리는 모두 **대화 순서**를 기준으로 저장·삭제해요. 최신 대화가 중요하고 수백 턴이 넘는 장기 대화에서는 오래된 정보를 아예 잃어버려요.

`VectorStoreRetrieverMemory`는 순서와 관계없이 **의미적 유사성**으로 검색해요. 100턴 전에 나눈 대화라도 현재 질문과 의미가 가깝다면 바로 꺼내올 수 있어요.

```mermaid
flowchart LR
    subgraph Sequential["순서 기반 메모리 (1~6번)"]
        S1[1턴] --> S2[2턴] --> S3[...] --> S100[100턴]
        S100 --> LATEST[최근 N턴만 사용]
    end

    subgraph Semantic["의미 기반 메모리 (이번 노트북)"]
        V1[1턴 벡터]
        V2[2턴 벡터]
        V100[100턴 벡터]
        Q[현재 질문] --"코사인 유사도"--> V2
        Q --"코사인 유사도"--> V100
    end

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    class S1,S2,S3 output
    class S100,LATEST output
    class V1,V2,V100 process
    class Q input
```

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드 및 벡터 스토어 메모리 관련 모듈 임포트
# ---------------------------------------------------
from dotenv import load_dotenv
from langchain.memory import VectorStoreRetrieverMemory
# OpenAIEmbeddings: 텍스트를 벡터로 변환하는 임베딩 모델
from langchain_openai import OpenAIEmbeddings
# FAISS: Meta가 만든 인메모리 벡터 검색 라이브러리
from langchain_community.vectorstores import FAISS

load_dotenv()

## 1. 기본 사용법 - 벡터 스토어 메모리 생성

먼저 벡터 스토어를 초기화하고, `VectorStoreRetrieverMemory`를 생성합니다.


### 💡 VectorStoreRetrieverMemory 초기화 흐름

```mermaid
graph TD
    LOAD_ENV --> EMBED[OpenAIEmbeddings()]
    EMBED --> VS[FAISS.from_texts([""], embeddings)]
    VS --> RETR[vectorstore.as_retriever(k=1)]
    RETR --> MEM_INIT[VectorStoreRetrieverMemory(retriever)]
```



In [ ]:
# ---------------------------------------------------
# 임베딩 모델 → FAISS 벡터 스토어 → Retriever → 메모리 순서로 초기화
# ---------------------------------------------------

# ============================================================
# TODO: OpenAIEmbeddings, FAISS, Retriever, VectorStoreRetrieverMemory를 순서대로 생성하세요
# 힌트:
#   1. embeddings = OpenAIEmbeddings()
#   2. vectorstore = FAISS.from_texts([""], embeddings)  # 빈 텍스트로 초기화
#   3. retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
#   4. memory = VectorStoreRetrieverMemory(retriever=retriever)
# 예상 결과: memory.save_context() 호출 시 대화가 벡터로 변환되어 FAISS에 저장됨
# ============================================================

# 1단계: 임베딩 모델 생성
# - OpenAIEmbeddings: 텍스트를 고차원 벡터로 변환 (의미적 유사성 계산에 사용)


# 2단계: 벡터 스토어 초기화 (빈 상태로 시작)
# - FAISS.from_texts([""], embeddings): 빈 텍스트로 인덱스를 초기화
# - 주의: FAISS는 인메모리 저장소로 서버 재시작 시 데이터가 사라짐


# 3단계: Retriever 생성 (상위 k개 문서 검색)
# - k=1: 가장 유사한 대화 1개만 반환 (토큰 절약)


# 4단계: VectorStoreRetrieverMemory 생성
# - retriever를 기반으로 save_context()/load_memory_variables() 구현


### 💡 벡터 스토어에 대화 저장

```mermaid
flowchart TD
    subgraph SaveLoop
        H[사용자 발화] --> SAVE[save_context]
        A[AI 응답] --> SAVE
    end

    SAVE --> EMBED_SAVE[임베딩 변환]
    EMBED_SAVE --> VS_UPSERT[FAISS 벡터 스토어에 upsert]
```



### 대화 저장 및 의미 기반 검색

대화를 저장하고, 질문에 따라 의미적으로 관련된 메모리를 검색합니다.


### 💡 의미 기반 검색 루프 (k=1)

```mermaid
flowchart LR
    QUESTIONS[[질문 리스트]] --> LOOP[for question in questions]
    LOOP --> EMBED_Q[질문 임베딩]
    EMBED_Q --> RETRIEVE[retriever.get_relevant_documents]
    RETRIEVE --> MEMORY_LOAD["load_memory_variables({'human': question})"]
    MEMORY_LOAD --> PRINT_RESULT[관련 history 출력]
```



In [ ]:
# ---------------------------------------------------
# 여러 대화를 벡터 스토어에 저장
# ---------------------------------------------------
# save_context() 호출 시 대화가 임베딩 벡터로 변환되어 FAISS 인덱스에 추가됨
# 순서가 아닌 의미적 유사도 기반으로 나중에 검색됨
conversations = [
    {
        "human": "안녕하세요! 오늘 프로젝트 회의에 참석해주셔서 감사합니다. 자기소개 부탁드립니다.",
        "ai": "안녕하세요. 저는 데이터 과학을 전공한 시니어 데이터 분석가입니다. 주로 파이썬과 SQL을 사용하며, 머신러닝 모델 개발과 데이터 시각화에 전문성을 가지고 있습니다."
    },
    {
        "human": "프로젝트에서 어떤 역할을 맡으실 계획인가요?",
        "ai": "제가 맡을 역할은 데이터 파이프라인 구축과 분석 모델 개발입니다. 고객 데이터를 수집하고 처리하여 인사이트를 도출하는 작업을 담당할 예정입니다. 또한 팀원들과 협력하여 분석 결과를 시각화하고 보고서를 작성하겠습니다."
    },
    {
        "human": "팀 프로젝트에서 어려움을 겪었던 경험이 있다면 어떻게 해결하셨나요?",
        "ai": "이전 프로젝트에서 데이터 품질 문제로 어려움을 겪었습니다. 이를 해결하기 위해 데이터 검증 프로세스를 도입하고, 팀원들과 정기적인 데이터 리뷰 미팅을 가졌습니다. 또한 자동화된 데이터 품질 체크 도구를 개발하여 문제를 사전에 발견할 수 있도록 했습니다."
    },
    {
        "human": "데이터 분석가로서 자신의 강점은 무엇이라고 생각하시나요?",
        "ai": "제 강점은 복잡한 데이터를 이해하기 쉬운 인사이트로 변환하는 능력입니다. 통계적 분석과 머신러닝 기법을 활용하여 숨겨진 패턴을 발견하고, 이를 비즈니스 의사결정에 도움이 되는 정보로 제공할 수 있습니다. 또한 팀워크를 중시하며 다양한 이해관계자와 효과적으로 소통합니다."
    }
]

# 모든 대화 저장 (각 대화가 임베딩 벡터로 변환되어 FAISS에 추가됨)


### 💡 k 값 변경 후 검색 흐름 (k=2)

```mermaid
flowchart TD
    RETR_INIT[기존 retriever(k=1)] -->|search_kwargs={'k':2}| RETR_K2[새 retriever]
    RETR_K2 --> MEM_K2[VectorStoreRetrieverMemory]
    MEM_K2 --> SAVE_K2[동일 대화 저장]
    SAVE_K2 --> RETRIEVE_K2["load_memory_variables({'human': 질문})"]
    RETRIEVE_K2 --> OUTPUT_K2[상위 2개 문서 history]
```



### 의미 기반 메모리 검색

질문에 따라 의미적으로 가장 관련성 높은 메모리를 검색합니다.


### 💡 순서 기반 vs 의미 기반 비교 흐름

```mermaid
flowchart TD
    DATASEQ[[test_conversations]] --> BUF_SAVE[buffer_memory.save_context]
    DATASEQ --> VEC_SAVE[vector_memory.save_context]

    BUF_SAVE --> BUF_HISTORY[history = 전체 메시지]
    VEC_SAVE --> VEC_INDEX[벡터 스토어 인덱스]

    BUF_HISTORY --> BUF_LOAD["load_memory_variables({})"]
    VEC_INDEX --> VEC_LOAD["load_memory_variables({'human': '두 번째'})"]

    BUF_LOAD --> REPORT_BUF[모든 메시지 수 출력]
    VEC_LOAD --> REPORT_VEC[질문과 의미적으로 가까운 문서만 반환]
```



In [ ]:
# ---------------------------------------------------
# 의미 기반 메모리 검색 (k=1)
# ---------------------------------------------------
# load_memory_variables({"human": 질문}): 질문을 임베딩으로 변환 후 코사인 유사도로 검색
# 저장 순서와 관계없이 의미적으로 가장 가까운 대화를 반환
questions = [
    "데이터 분석가의 전공은 무엇인가요?",
    "프로젝트에서 맡을 역할은 무엇인가요?",
    "어려움을 어떻게 해결했나요?",
    "강점은 무엇인가요?"
]


## 2. k 값 조정하기

`k` 값을 조정하여 더 많은 관련 메모리를 검색할 수 있습니다.


## 3. 다른 메모리 타입과의 차이

`VectorStoreRetrieverMemory`는 대화 순서를 추적하지 않고 의미 기반으로 검색합니다.


In [ ]:
# ---------------------------------------------------
# ConversationBufferMemory(순서 기반) vs VectorStoreRetrieverMemory(의미 기반) 비교
# ---------------------------------------------------
# - BufferMemory: 대화를 순서대로 저장, load 시 전체 반환
# - VectorStoreRetrieverMemory: 벡터로 저장, load 시 질문과 유사한 k개만 반환

from langchain.memory import ConversationBufferMemory

# 비교용 대화
test_conversations = [
    {"human": "첫 번째 대화입니다.", "ai": "첫 번째 답변입니다."},
    {"human": "두 번째 대화입니다.", "ai": "두 번째 답변입니다."},
    {"human": "세 번째 대화입니다.", "ai": "세 번째 답변입니다."},
]

# ConversationBufferMemory: 순서대로 저장 (제한 없음)


# VectorStoreRetrieverMemory: 의미 기반 검색 (k=1)


# 비교


## 핵심 정리

```python
# VectorStoreRetrieverMemory 기본 사용법
from langchain.memory import VectorStoreRetrieverMemory
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 벡터 스토어 초기화
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts([""], embeddings)

# Retriever 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# 메모리 생성
memory = VectorStoreRetrieverMemory(retriever=retriever)

# 대화 저장
memory.save_context(
    inputs={"human": "사용자 메시지"},
    outputs={"ai": "AI 응답"}
)

# 의미 기반 검색
memory_vars = memory.load_memory_variables({"human": "질문"})
```

### 주요 특징
- ✅ **의미 기반 검색**: 벡터 임베딩을 사용하여 의미적으로 관련된 메모리 검색
- ✅ **순서 무관**: 대화 순서를 명시적으로 추적하지 않음
- ✅ **장기 메모리**: 대량의 메모리를 효율적으로 저장하고 검색 가능
- ✅ **관련성 기반**: 질문과 관련성 높은 메모리만 반환

### 다른 메모리 타입과의 차이

| 특징 | ConversationBufferMemory | VectorStoreRetrieverMemory |
|------|-------------------------|---------------------------|
| 저장 방식 | 순서대로 모든 대화 저장 | 벡터 스토어에 저장 |
| 검색 방식 | 순서 기반 | 의미 기반 (벡터 검색) |
| 반환 내용 | 모든 대화 | 관련성 높은 k개만 |
| 대화 순서 | 중요함 | 중요하지 않음 |
| 적합한 경우 | 짧은 대화, 순서 중요 | 긴 대화, 의미적 관련성 중요 |

### 주의사항
- ⚠️ **벡터 스토어 필요**: FAISS, Chroma 등의 벡터 스토어가 필요함
- ⚠️ **임베딩 모델**: 임베딩 모델의 성능에 따라 검색 품질이 달라짐
- ⚠️ **k 값 선택**: 너무 작으면 정보 부족, 너무 크면 노이즈 증가
- ⚠️ **순서 정보 손실**: 대화 순서 정보가 손실될 수 있음